In [1]:
import pandas as pd 

In [2]:
df = pd.read_csv("../data/perfumes.csv",
    sep=";", # separador de columnas
    encoding="cp1252", # codificacion de caracteres
    decimal="," # separador decimal
)
df.head()

,url,Perfume,Brand,Country,Gender,Rating Value,Rating Count,Year,Top,Middle,Base,Perfumer1,Perfumer2,mainaccord1,mainaccord2,mainaccord3,mainaccord4,mainaccord5
0,https://www.fragrantica.com/perfume/xerjoff/ac...,accento-overdose-pride-edition,xerjoff,Italy,unisex,1.42,201,2022.0,"fruity notes, aldehydes, green notes","bulgarian rose, egyptian jasmine, lily-of-the-...","eucalyptus, pine",unknown,NaN,rose,woody,fruity,aromatic,floral
1,https://www.fragrantica.com/perfume/jean-paul-...,classique-pride-2024,jean-paul-gaultier,France,women,1.86,70,2024.0,"yuzu, citruses","orange blossom, neroli","musk, blonde woods",unknown,NaN,citrus,white floral,sweet,fresh,musky
2,https://www.fragrantica.com/perfume/jean-paul-...,classique-pride-2023,jean-paul-gaultier,France,unisex,1.91,285,2023.0,"blood orange, yuzu","neroli, orange blossom","musk, white woods",natalie gracia-cetto,quentin bisch,citrus,white floral,sweet,fresh spicy,musky
3,https://www.fragrantica.com/perfume/bruno-bana...,pride-edition-man,bruno-banani,Germany,men,1.92,59,2019.0,"guarana, grapefruit, red apple","walnut, lavender, guava","vetiver, benzoin, amber",unknown,NaN,fruity,nutty,woody,tropical,NaN
4,https://www.fragrantica.com/perfume/jean-paul-...,le-male-pride-collector,jean-paul-gaultier,France,men,1.93,632,2020.0,"mint, lavender, cardamom, artemisia, bergamot","caraway, cinnamon, orange blossom","vanilla, sandalwood, amber, cedar, tonka bean",francis kurkdjian,NaN,aromatic,warm spicy,fresh spicy,cinnamon,vanilla


In [3]:
# eliminar columnas que no voy a utilizar
df = df.drop(columns=["url","Year","Perfumer1","Perfumer2","Rating Value"])

df.shape

(24063, 13)

In [4]:
# comprobar los valores nulos por columna
df.isnull().sum()

Perfume           0
Brand             0
Country           0
Gender            0
Rating Count      0
Top               0
Middle            0
Base              0
mainaccord1       0
mainaccord2      13
mainaccord3     114
mainaccord4     388
mainaccord5     981
dtype: int64

In [5]:
# contar las filas duplicadas
df.duplicated().sum()

np.int64(0)

In [6]:
# comprobar las categorias de los perfumes
df["Gender"].value_counts()

Gender
women     11376
unisex     7698
men        4989
Name: count, dtype: int64

***

In [7]:
# contar los paises diferentes
df["Country"].nunique()

54

In [8]:
# mostrar todos los paises ordenados alfabeticamente
for pais in sorted(df["Country"].unique()):
    print(pais)

Arabia saudi 
Argentina
Australia
Bahrain
Bahrein
Belgium
Bermuda
Brazil
Canada
China
Czech Republic
Denmark
Egypt
Finland
France
Germany
Greece
Hong Kong
Hungary
Iceland
India
Indonesia
Iran
Ireland
Israel
Italy
Japan
Kazakhstan
Kuwait
Latvia
Lebanon
Malaysia
Mexico
Monaco
Morocco
Netherlands
Oman
Pakistan
Peru
Poland
Portugal
Qatar
Romania
Russia
South Korea
Spain
Sweden
Switzerland
Thailand
Turkey
UAE
UK
USA
Ukraine


In [9]:
# normalizar los nombres de paises
df["Country"] = df["Country"].replace({
    "Bahrein": "Bahrain",
    "Arabia saudi": "Saudi Arabia"
})

***

In [10]:
# revisar todos los acordes principales y sus frecuencias
print(df["mainaccord1"].value_counts().to_string())

mainaccord1
citrus             3320
woody              3152
fruity             2280
white floral       1937
floral             1682
aromatic           1666
warm spicy         1329
amber              1273
sweet              1225
vanilla             987
rose                823
powdery             802
fresh spicy         547
green               503
musky               405
leather             277
oud                 159
aquatic             134
fresh               128
marine              119
iris                105
yellow floral        97
ozonic               94
lavender             87
patchouli            80
tropical             77
almond               63
earthy               51
caramel              49
soft spicy           49
tobacco              48
coconut              46
honey                45
violet               41
tuberose             40
cherry               39
lactonic             38
aldehydic            36
cacao                26
smoky                25
coffee               22
moss

In [ ]:
# detectar posibles errores ortograficos
%pip install pyspellchecker

In [12]:
from spellchecker import SpellChecker

# crear un corrector ortografico en ingles
spell = SpellChecker(language="en")

In [13]:
columnas_acordes = [
    "mainaccord1",
    "mainaccord2",
    "mainaccord3",
    "mainaccord4",
    "mainaccord5"
]

# obtener todos los acordes diferentes
acordes = pd.unique(df[columnas_acordes].values.ravel())

# eliminar los valores nulos
acordes = pd.Series(acordes).dropna()

In [14]:
# buscar palabras que el corrector no reconoce
for acorde in acordes:
    for palabra in acorde.split():
        if palabra in spell.unknown([palabra]):
            print(palabra, "→", spell.correction(palabra))

animalic → animal
aldehydic → aldehyde
lactonic → laconic
terpenic → herpetic
coca-cola → None
asphault → asphalt


In [15]:
# corregir el error detectado
df[columnas_acordes] = df[columnas_acordes].replace({
    "asphault": "asphalt"
})

In [16]:
# eliminar espacios innecesarios en los acordes
for columna in columnas_acordes:
    df[columna] = df[columna].str.strip()

***

In [18]:
df.to_csv("../data/perfumes_limpio.csv", index=False)